In [1]:
import csv , pandas as pd 
import psycopg2
from psycopg2.extras import execute_values
import numpy as np
import io
from datetime import datetime
from uuid import UUID
import random
import re
from sshtunnel import SSHTunnelForwarder
from dotenv import load_dotenv
import os
import subprocess

In [2]:
load_dotenv()
env = os.environ

SSH_HOST = env.get("SSH_HOST")
SSH_PORT = int(env.get("SSH_PORT", 22))
SSH_USERNAME = env.get("SSH_USERNAME")
SSH_PASSWORD = env.get("SSH_PASSWORD")
SSH_PRIVATE_KEY = env.get("SSH_PRIVATE_KEY")


DB_HOST = env.get("DB_HOST")
DB_PORT = int(env.get("DB_PORT", 5432))
DB_NAME = env.get("DB_NAME")
DB_USER = env.get("DB_USER")
DB_PASSWORD = env.get("DB_PASSWORD")

# ======================================================
# Create SSH Tunnel
# ======================================================

tunnel_args = {
    "ssh_address_or_host": (SSH_HOST, SSH_PORT),
    "ssh_username": SSH_USERNAME,
    "remote_bind_address": (DB_HOST, DB_PORT),
    "allow_agent": False,
    "host_pkey_directories": [],
}

if SSH_PASSWORD:
    tunnel_args["ssh_password"] = SSH_PASSWORD

if SSH_PRIVATE_KEY and os.path.isfile(SSH_PRIVATE_KEY):
    tunnel_args["ssh_pkey"] = SSH_PRIVATE_KEY

tunnel = SSHTunnelForwarder(**tunnel_args)
tunnel.start()

print(f"✓ SSH Tunnel Established (localhost:{tunnel.local_bind_port})")

# ======================================================
# PostgreSQL Connection
# ======================================================

conn = None

try:
    conn = psycopg2.connect(
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
        host="127.0.0.1",
        port=tunnel.local_bind_port,
    )
    print("✓ Database connected successfully!")

except psycopg2.Error as e:
    print(f"Database connection error: {e}")

✓ SSH Tunnel Established (localhost:19607)
✓ Database connected successfully!


In [3]:
kmat = pd.read_csv('KMAT.csv',delimiter="\t")
kmat

,Old material no.,Material description,Language,New Material
0,001A,LT-25 - FANUC,EN,40001
1,001B,LT-25 - SIEMENS,EN,40002
2,002A,LT-25 630 - FANUC,EN,40003
3,003A,LT-20 C CLASSIC - FANUC,EN,40004
4,003B,LT-20 C CLASSIC - SIEMENS,EN,40005
...,...,...,...,...
577,M36,FNX630,EN,40712
578,M37,MCH500C,EN,40564
579,M38,MCH630XL,EN,40569
580,M39,MCV450-PLUS-RAPC,EN,40614


In [ ]:
# input_file = r"24082026_cleanups\05_dq_out_of_kmat_cnc.csv"
# filtered_file = "event_logs_25Aug2026.csv"

# # Remove existing index column
# if "index" in df.columns:
#     df = df.drop(columns=["index"])

# # Filter
# filtered = df[
#     df["SupplierProduct"].astype(str).str.strip() == "MCV450 PLUS"
# ]

# # Save without index
# filtered.to_csv(
#     filtered_file,
#     index=False,
#     quoting=csv.QUOTE_ALL
# )

# print(f"Moved {len(filtered)} records to {filtered_file}")


In [4]:
with open('event_log_master_sap_10Oct2026.csv', 'r', encoding='utf-8', errors='replace') as f:
    content = f.read().replace('Sector" B"', 'Sector B')

df = pd.read_csv(
    io.StringIO(content),
    delimiter=",",
    engine="python"
)

print("Total records in df : ", len(df))  # Outputs: 601


Total records in df :  1


In [ ]:
import pandas as pd

df = pd.read_csv(
    '28072026_cleanups/12092026_cleanups_v1/05_dq_out_of_kmat_cnc.csv',
    delimiter=",",
    encoding='utf-8',
    engine="python"
)

print("Total records in df : ", len(df))


In [ ]:
df[df["ActivationKey"].notna()]

In [5]:
# Clean whitespace first
df['SupplierProduct'] = df['SupplierProduct'].astype(str).str.strip()
df['CustomerProductSlno'] = df['CustomerProductSlno'].astype(str).str.strip()

# Dictionary of specific product name overrides
product_name_overrides = {
    'J 300 LM': 'J 300 LM - FANUC',
    'LT 2 LM 800': 'LT-2 LM 800 - FANUC',
    'LT 20 C CLASSIC': 'LT-20 C CLASSIC - FANUC',
    'LT-12/CUB LM': 'LT-12 / CUB LM - FANUC',
    'LT-12/CUBLM': 'LT-12 / CUB LM - FANUC',
    'LT-20 C CLASSIC': 'LT-20 C CLASSIC - FANUC',
    'MCV450XL PLUS': 'MCV450 PLUS',
    'SIMPLE TURN 50125 XL': 'SIMPLETURN 5075 XL - FANUC',
    'SLT 06 LM': 'SLT 06 LM - FANUC',
    'SLT 08 RG': 'SLT 08 RG - FANUC',
    'WLH - 22': 'WLH 22 - FANUC',
    'WLH 22': 'WLH 22 - FANUC',
    'MCV450-NEW-LAPC':'MCV450-PLUS-LAPC'
}

# Apply replacements
df['SupplierProduct'] = df['SupplierProduct'].replace(product_name_overrides)

# Summary of updated rows
updated_count = df['SupplierProduct'].isin(product_name_overrides.values()).sum()
print(f"✓ Successfully mapped manual replacements! Matching records updated: {updated_count}")


✓ Successfully mapped manual replacements! Matching records updated: 1


In [6]:
# 1. segregate out Cancelled Machines
cancelled_df = df[df['CancelIndicator'] == 'X']
cancelled_df.to_csv('01_dq_cancelled_machines.csv',index=False,quoting=csv.QUOTE_ALL)

df = df[df['CancelIndicator'].isna()].reset_index(drop=True)
print("Total records : ",len(df))

Total records :  1


In [7]:
# 2. find duplicates of cnc id machines
duplicates_cnc = df[df.duplicated(subset=['CNCID'],keep=False) & df['CNCID'].notna()]
duplicates_cnc.to_csv('02_dq_duplicated_cnc.csv',index=False,quoting=csv.QUOTE_ALL)

null_cnc = df[df['CNCID'].isnull()]
null_cnc.to_csv('03_dq_null_cnc.csv',index=False,quoting=csv.QUOTE_ALL)


df = df[df['CNCID'].notna() & ~df.duplicated(subset=['CNCID'],keep=False)].reset_index(drop=False)

print("Total records : ",len(df))

Total records :  1


In [8]:
# 3. Invalid CNCID
pattern = r'^[^-]{8}(?:-[^-]{8}){3}$'

invalid_cnc_id = df[~df["CNCID"].astype(str).str.match(pattern, na=False)]

df = df[
    df["CNCID"].astype(str).str.match(pattern, na=False)
].copy()

invalid_cnc_id.to_csv('04_dq_invalid_cncid_pattern.csv',sep=',',quoting=csv.QUOTE_ALL,index=False)

print("Total records : ",len(df))

Total records :  1


In [9]:
# 3b . Check that cnc_id is already present in sap.fact_machine_dispatch_logs
query = """
SELECT DISTINCT cnc_id 
FROM sap.fact_machine_dispatch_logs 
WHERE cnc_id IS NOT NULL;
"""

with conn.cursor() as cur:
    cur.execute(query)
    existing_cnc_ids = set(str(row[0]).strip() for row in cur.fetchall() if row[0] is not None)

already_present_cnc_id = df[df["CNCID"].astype(str).str.strip().isin(existing_cnc_ids)]

df = df[~df["CNCID"].astype(str).str.strip().isin(existing_cnc_ids)].copy()

already_present_cnc_id.to_csv('04b_dq_already_present_cncid.csv', sep=',', quoting=csv.QUOTE_ALL, index=False)

print("Total records : ", len(df))


Total records :  1


In [10]:
# Array of unique values
df['SupplierProduct'].unique()


<StringArray>
['J 300 LM - FANUC']
Length: 1, dtype: str

In [11]:
# Helper function to normalize text (strips spaces, hyphens, slashes, uppercase)
def normalize_key(val):
    return re.sub(r'[^A-Z0-9]', '', str(val).upper()) if pd.notna(val) and str(val).strip() != "" else ""

# ------------------------------------------------------------------
# 1. Build KMAT Lookups (EXCLUDING Old material no.)
# ------------------------------------------------------------------
# Primary: New Material (5-digit code -> Material description)
newmat_lookup = dict(
    zip(
        kmat['New Material'].dropna().astype(str).str.strip(),
        kmat['Material description'].dropna().astype(str).str.strip()
    )
)

# Fallback: Material description ONLY
desc_lookup = {}
for _, row in kmat.dropna(subset=['Material description']).iterrows():
    mat_desc = str(row['Material description']).strip()
    norm_desc = normalize_key(mat_desc)
    
    if norm_desc and norm_desc not in desc_lookup:
        desc_lookup[norm_desc] = mat_desc

# ------------------------------------------------------------------
# 2. Match df Rows (Read-Only Copy)
# ------------------------------------------------------------------
temp_view = df[['SupplierProduct', 'CustomerProductSlno']].copy()
temp_view['SupplierProduct'] = temp_view['SupplierProduct'].astype(str).str.strip()
temp_view['CustomerProductSlno'] = temp_view['CustomerProductSlno'].astype(str).str.strip()

# Pass 1: FIRST check New Material (Prefix before '-')
has_hyphen = temp_view['CustomerProductSlno'].str.contains('-', na=False)
slno_prefix = temp_view['CustomerProductSlno'].str.split('-').str[0].str.strip()

temp_view['Matched_KMAT_Name'] = None
temp_view['Match_Type'] = None

# Match Pass 1
prefix_matched = slno_prefix.where(has_hyphen).map(newmat_lookup)
temp_view['Matched_KMAT_Name'] = prefix_matched
temp_view.loc[temp_view['Matched_KMAT_Name'].notna(), 'Match_Type'] = 'New Material Prefix Match'

# Pass 2: SECOND / FALLBACK check by normalized Material description
fallback_mask = temp_view['Matched_KMAT_Name'].isna()
temp_view['norm_key'] = temp_view['SupplierProduct'].apply(normalize_key)

temp_view.loc[fallback_mask, 'Matched_KMAT_Name'] = temp_view.loc[fallback_mask, 'norm_key'].map(desc_lookup)
temp_view.loc[fallback_mask & temp_view['Matched_KMAT_Name'].notna(), 'Match_Type'] = 'Material Description Match'

# ------------------------------------------------------------------
# 3. Summary Table
# ------------------------------------------------------------------
matched_records = temp_view[temp_view['Matched_KMAT_Name'].notna()]

summary = (
    matched_records.groupby('Matched_KMAT_Name')
    .agg(
        Total_Rows_in_Data=('SupplierProduct', 'count'),
        Variations_in_Data=('SupplierProduct', lambda x: sorted(list(set(x)))),
        Match_Methods=('Match_Type', lambda x: sorted(list(set(x))))
    )
    .reset_index()
)

summary['Variation_Count'] = summary['Variations_in_Data'].apply(len)

# Sort: Most variations first, then highest row count
summary = summary[
    ['Matched_KMAT_Name', 'Total_Rows_in_Data', 'Variation_Count', 'Variations_in_Data', 'Match_Methods']
].sort_values(by=['Variation_Count', 'Total_Rows_in_Data'], ascending=[False, False]).reset_index(drop=True)

# Display Table
summary


,Matched_KMAT_Name,Total_Rows_in_Data,Variation_Count,Variations_in_Data,Match_Methods
0,J 300 LM - FANUC,1,1,[J 300 LM - FANUC],[New Material Prefix Match]


In [12]:
output_file = 'unmatched_kmat_records.csv'

unmatched_summary = (
    temp_view[temp_view['Matched_KMAT_Name'].isna()]
    .groupby(['SupplierProduct', 'CustomerProductSlno'])
    .size()
    .reset_index(name='Total_Rows')
    .sort_values(by='Total_Rows', ascending=False)
)
unmatched_summary.to_csv(output_file, index=False)
print(f"✓ Saved distinct summary to '{output_file}'")


✓ Saved distinct summary to 'unmatched_kmat_records.csv'


In [13]:
def normalize_key(val):
    return re.sub(r'[^A-Z0-9]', '', str(val).upper()) if pd.notna(val) and str(val).strip() != "" else ""

# 1. KMAT Lookups (Excluding Old material no.)
# Primary: New Material (5-digit code -> Material description)
newmat_lookup = dict(
    zip(
        kmat['New Material'].dropna().astype(str).str.strip(),
        kmat['Material description'].dropna().astype(str).str.strip()
    )
)

# Fallback: Normalized Material description
desc_lookup = {
    normalize_key(desc): str(desc).strip()
    for desc in kmat['Material description'].dropna().unique()
    if normalize_key(desc)
}

# 2. Match df['SupplierProduct']
# Step A: FIRST match via CustomerProductSlno prefix (before '-')
has_hyphen = df['CustomerProductSlno'].astype(str).str.contains('-', na=False)
slno_prefix = df['CustomerProductSlno'].astype(str).str.split('-').str[0].str.strip()
prefix_matched_names = slno_prefix.where(has_hyphen).map(newmat_lookup)

# Step B: SECOND / FALLBACK match via normalized Material description
norm_series = df['SupplierProduct'].apply(normalize_key)
desc_matched_names = norm_series.map(desc_lookup)

# 3. Apply corrected names (New Material first -> Material desc fallback -> Original)
df['SupplierProduct'] = (
    prefix_matched_names
    .fillna(desc_matched_names)
    .fillna(df['SupplierProduct'])
)

print("✓ SupplierProduct names standardized successfully!")


✓ SupplierProduct names standardized successfully!


In [14]:
# 4. Update Supplier 'ACE' -> 'MCD' and  'AMS' -> 'TCD'
df['Supplier'] = df['Supplier'].replace({
    'ACE':'MCD',
    'AMS':'TCD'
})

In [15]:
original_df = df.copy()

# ------------------------------------------------------------------
# 1. Clean input columns
# ------------------------------------------------------------------
df['SupplierProduct'] = df['SupplierProduct'].astype(str).str.strip()
df['CustomerProductSlno'] = df['CustomerProductSlno'].astype(str).str.strip()

# ------------------------------------------------------------------
# 2. KMAT lookup tables
# ------------------------------------------------------------------
kmat_desc_lookup = kmat[
    ['Material description', 'Old material no.', 'New Material']
].drop_duplicates(subset=['Material description'])

kmat_newmat_lookup = kmat[
    ['Material description', 'New Material', 'Old material no.']
].drop_duplicates(subset=['New Material'])

# ------------------------------------------------------------------
# 3. FIRST: Direct match on Material Description
# ------------------------------------------------------------------
direct_df = df.merge(
    kmat_desc_lookup,
    left_on='SupplierProduct',
    right_on='Material description',
    how='inner'
)

direct_df['MappingType'] = 0

# ------------------------------------------------------------------
# 4. Keep only unmatched rows
# ------------------------------------------------------------------
remaining_df = df[
    ~df['SupplierProduct'].isin(direct_df['SupplierProduct'])
].copy()

# ------------------------------------------------------------------
# 5. Existing logic starts here (only on unmatched rows)
# ------------------------------------------------------------------
kmat_descriptions = set(
    kmat['Material description'].dropna().astype(str).str.strip()
)
kmat_new_materials = set(
    kmat['New Material'].dropna().astype(str).str.strip()
)

supplier_prod = remaining_df['SupplierProduct'].astype(str).str.strip()
customer_slno = remaining_df['CustomerProductSlno'].astype(str).str.strip()

has_hyphen = customer_slno.str.contains('-', na=False)
slno_prefix = customer_slno.str.split('-').str[0].str.strip()

match_description = supplier_prod.isin(kmat_descriptions)
match_new_material = has_hyphen & slno_prefix.isin(kmat_new_materials)

# Keep only rows matching KMAT
remaining_df = remaining_df[
    match_description | match_new_material
].reset_index(drop=True)

# Prepare keys
remaining_df['SupplierProduct'] = remaining_df['SupplierProduct'].astype(str).str.strip()
remaining_df['CustomerProductSlno'] = remaining_df['CustomerProductSlno'].astype(str).str.strip()
remaining_df['Prefix'] = remaining_df['CustomerProductSlno'].str.split('-').str[0]

# Split into two groups
desc_df = remaining_df[
    ~remaining_df['CustomerProductSlno'].str.contains('-', na=False)
].copy()

newmat_df = remaining_df[
    remaining_df['CustomerProductSlno'].str.contains('-', na=False)
].copy()

# Merge by Material description
desc_df = desc_df.merge(
    kmat_desc_lookup,
    left_on='SupplierProduct',
    right_on='Material description',
    how='left',
)

# Merge by New Material
newmat_df = newmat_df.merge(
    kmat_newmat_lookup,
    left_on='Prefix',
    right_on='New Material',
    how='left'
)

# Mapping type indicator
desc_df['MappingType'] = 0
newmat_df['MappingType'] = 1

# ------------------------------------------------------------------
# 6. Final result
# ------------------------------------------------------------------
df = pd.concat(
    [direct_df, desc_df, newmat_df],
    ignore_index=True
)

df.drop(columns=['Prefix'], inplace=True, errors='ignore')

# Use official KMAT Material description
df['SupplierProduct'] = df['Material description'].fillna(df['SupplierProduct'])

print("Total records:", len(df))

Total records: 1


In [16]:
dummy_df = original_df[
    ~original_df["CNCID"].isin(df["CNCID"])
].copy()

dummy_df.to_csv("05_dq_out_of_kmat_cnc.csv",index=False,sep=",",quoting=csv.QUOTE_ALL)

# Free memory
del original_df

In [17]:
# 6. feature hardcdoed
df['Features'] = 10000000

In [18]:
# 7. updated Invoice and InvoiceItem with random Values
df['Invoice'] = df['Invoice'].astype('string')
df['InvoiceItem'] = df['InvoiceItem'].astype('string')
used = set(df['Invoice'].dropna().astype(str))

def random_invoice():
    while True:
        inv = str(random.randint(100, 999))
        if inv not in used:
            used.add(inv)
            return inv

invoice_updates = {}
invoiceitem_updates = {}

for (customer, cncid), grp in df.groupby(['Customer', 'CNCID'], sort=False):

    # -------- Invoice --------
    inv = None

    # Reuse existing invoice if present in this CNC group
    existing = grp['Invoice'].dropna()
    if not existing.empty:
        inv = str(existing.iloc[0])
    else:
        inv = random_invoice()

    # Fill only missing invoices
    for idx in grp[grp['Invoice'].isna()].index:
        invoice_updates[idx] = inv

    # -------- InvoiceItem --------
    item_rows = grp[grp['InvoiceItem'].isna()].index

    for i, idx in enumerate(item_rows, start=1):
        invoiceitem_updates[idx] = str(i * 10)

# Bulk update (very fast)
df.loc[invoice_updates.keys(), 'Invoice'] = list(invoice_updates.values())
df.loc[invoiceitem_updates.keys(), 'InvoiceItem'] = list(invoiceitem_updates.values())

In [20]:
# 8. update sap.dim_product table first

now = datetime.now()
zero_uuid = "00000000-0000-0000-0000-000000000000"

# 1. Extract columns and rename
upsert_df = (
    df[
        [
            "Supplier",
            "Material description",
            "Old material no.",
            "New Material",
        ]
    ]
    .dropna(subset=["Material description"])
    .copy()
    .rename(
        columns={
            "Material description": "supplier_product",
            "Old material no.": "old_material_no",
            "New Material": "new_material_no",
        }
    )
)

# 2. Supplier mapping (MCD -> 1, TCD -> 2)
upsert_df["supplier_id"] = upsert_df["Supplier"].apply(
    lambda x: 1 if str(x).strip().upper() == "MCD" else 2
)

# 3. Deduplicate on (supplier_id, supplier_product) to match SQL ON CONFLICT target
upsert_df = upsert_df.drop_duplicates(
    subset=["supplier_id", "supplier_product"]
)

# 4. Clean strings and handle nulls
for col in ["supplier_product", "old_material_no", "new_material_no"]:
  upsert_df[col] = (
      upsert_df[col]
      .astype(str)
      .str.strip()
      .replace({"nan": None, "None": None, "": None})
  )

# 5. Audit columns
upsert_df["created_by"] = zero_uuid
upsert_df["updated_by"] = zero_uuid
upsert_df["created_at"] = now
upsert_df["updated_at"] = now
upsert_df["updated_type"] = "backend"

# 6. Final column order
upsert_df = upsert_df[
    [
        "supplier_id",
        "supplier_product",
        "created_by",
        "updated_by",
        "created_at",
        "updated_at",
        "updated_type",
        "old_material_no",
        "new_material_no",
    ]
]

# 7. Convert remaining NaNs -> None
upsert_df = upsert_df.where(upsert_df.notna(), None)

# 8. Convert to tuple list for SQL insertion
records = list(upsert_df.itertuples(index=False, name=None))

In [21]:
try:
    with conn.cursor() as cur:
        cur.execute("""
            ALTER TABLE sap.dim_product
            ADD COLUMN IF NOT EXISTS old_material_no text,
            ADD COLUMN IF NOT EXISTS new_material_no text;
        """)
    conn.commit()
    print("Columns added successfully.")
except Exception as e:
    conn.rollback()
    print(e)

Columns added successfully.


In [22]:
sql = """
INSERT INTO sap.dim_product (
    supplier_id,
    supplier_product,
    created_by,
    updated_by,
    created_at,
    updated_at,
    updated_type,
    old_material_no,
    new_material_no
)
VALUES %s
ON CONFLICT (supplier_id,supplier_product)
DO UPDATE SET
    supplier_id     = EXCLUDED.supplier_id,
    old_material_no = EXCLUDED.old_material_no,
    new_material_no = EXCLUDED.new_material_no,
    updated_by      = EXCLUDED.updated_by,
    updated_at      = EXCLUDED.updated_at,
    updated_type    = EXCLUDED.updated_type;
"""

with conn.cursor() as cur:
    execute_values(cur, sql, records, page_size=1000)

conn.commit()

In [24]:
# 9. Insert data into bronze layer

brz_df = df.copy()

# Keep only this Supplier Product
# brz_df = brz_df[
#     brz_df["supplier_product"].astype(str).str.strip() == "MCV700N+50"
# ].copy()

brz_df = brz_df.rename(
    columns={
        "Branch": "branch",
        "Location": "location",
        "Customer": "customer",
        "Supplier": "supplier",
        "SupplierProduct": "supplier_product",
        "CustomerProductSlno": "customer_product_slno",
        "DispatchDate": "dispatch_date",
        "CommissionedDate": "commissioned_date",
        "CustomerProductStatus": "customer_product_status",
        "Features": "features",
        "CNCID": "cnc_id",
        "UpdatedBy": "updated_by",
        "UpdatedTS": "updated_ts",
        "Invoice": "invoice",
        "CancelIndicator": "cancel_indicator",
        "InvoiceItem": "invoice_item",
        "CustomerNo": "customer_no",
        "CustomerAddress": "customer_address",
        "CustomerCity": "customer_city",
        "CustomerPincode": "customer_pincode",
        "ShipToNo": "ship_to_no",
        "ShipToName": "ship_to_name",
        "ShipToAddress": "ship_to_address",
        "ShipToCity": "ship_to_city",
        "ShipToPincode": "ship_to_pincode",
    }
)

# Convert dates & timestamps
brz_df["dispatch_date"] = pd.to_datetime(
    brz_df["dispatch_date"], errors="coerce"
).dt.date
brz_df["commissioned_date"] = pd.to_datetime(
    brz_df["commissioned_date"], errors="coerce"
).dt.date
brz_df["updated_ts"] = pd.to_datetime(brz_df["updated_ts"], errors="coerce")

# Add DB metadata columns
brz_df["job_id"] = 66
brz_df["ingested_ts"] = pd.Timestamp.now()
brz_df["created_at"] = pd.Timestamp.now()

# Define DB column order
db_columns = [
    "job_id",
    "branch",
    "location",
    "customer",
    "supplier",
    "supplier_product",
    "customer_product_slno",
    "dispatch_date",
    "commissioned_date",
    "customer_product_status",
    "features",
    "cnc_id",
    "updated_by",
    "updated_ts",
    "invoice",
    "cancel_indicator",
    "invoice_item",
    "customer_no",
    "customer_address",
    "customer_city",
    "customer_pincode",
    "ship_to_no",
    "ship_to_name",
    "ship_to_address",
    "ship_to_city",
    "ship_to_pincode",
    "ingested_ts",
    "created_at",
]

brz_df = brz_df[db_columns]
brz_df = brz_df.replace(r"^\s*$", np.nan, regex=True)
brz_df = brz_df.where(pd.notnull(brz_df), None)

print("brz_df shape:", brz_df.shape)
print("Sample supplier_product values:")
print(brz_df[["supplier_product", "customer_product_slno", "job_id"]].head())

brz_df shape: (1, 28)
Sample supplier_product values:
   supplier_product customer_product_slno  job_id
0  J 300 LM - FANUC           40209-14205      66


In [25]:
brz_df['supplier_product'].unique()

<StringArray>
['J 300 LM - FANUC']
Length: 1, dtype: str

In [26]:
brz_df.to_csv('bronze_df_20260825.csv',index=False,quoting = csv.QUOTE_ALL)

In [27]:
buffer = io.StringIO()
brz_df.to_csv(buffer, index=False, header=False, sep='\t', na_rep='\\N')
buffer.seek(0)

# 2. Define the COPY command
copy_sql = r"""
COPY bronze.raw_machine_dispatch_logs (
    job_id,
    branch,
    location,
    customer,
    supplier,
    supplier_product,
    customer_product_slno,
    dispatch_date,
    commissioned_date,
    customer_product_status,
    features,
    cnc_id,
    updated_by,
    updated_ts,
    invoice,
    cancel_indicator,
    invoice_item,
    customer_no,
    customer_address,
    customer_city,
    customer_pincode,
    ship_to_no,
    ship_to_name,
    ship_to_address,
    ship_to_city,
    ship_to_pincode,
    ingested_ts,
    created_at
) FROM STDIN WITH (FORMAT csv, DELIMITER E'\t', NULL '\N');
"""

# 3. Execute Bulk COPY
with conn.cursor() as cur:
    cur.copy_expert(sql=copy_sql, file=buffer)

conn.commit()

print("Bulk COPY inserted successfully!")


Bulk COPY inserted successfully!


In [28]:
# 10. Fetch existing Branch/Location combinations and update that sap.dim_branch
with conn.cursor() as cur:
    cur.execute("""
        SELECT branch_name, location_name
        FROM sap.dim_branch
    """)
    db_combos = {
        (branch.strip(), location.strip())
        for branch, location in cur.fetchall()
    }

# Get unique combinations from Excel
branch_df = (
    df[['Branch', 'Location', 'Region']]
    .dropna(subset=['Branch', 'Location'])
    .copy()
)

# Strip whitespace
branch_df["Branch"] = branch_df["Branch"].astype(str).str.strip()
branch_df["Location"] = branch_df["Location"].astype(str).str.strip()
branch_df["Region"] = branch_df["Region"].astype(str).str.strip()

# Remove duplicates after stripping
branch_df = branch_df.drop_duplicates(subset=["Branch", "Location"])

# Keep only rows not present in DB
missing_df = branch_df[
    ~pd.MultiIndex.from_frame(
        branch_df[["Branch", "Location"]]
    ).isin(db_combos)
].copy()

print(f"Rows to insert: {len(missing_df)}")

# Add audit columns
missing_df["created_by"] = zero_uuid
missing_df["updated_by"] = zero_uuid
missing_df["created_at"] = now
missing_df["updated_at"] = now
missing_df["updated_type"] = "service"

# Create records
records = list(
    missing_df[
        [
            "Branch",
            "Location",
            "created_by",
            "updated_by",
            "created_at",
            "updated_at",
            "Region",
            "updated_type",
        ]
    ].itertuples(index=False, name=None)
)

Rows to insert: 1


In [29]:
sql = """
INSERT INTO sap.dim_branch
(
    branch_name,
    location_name,
    created_by,
    updated_by,
    created_at,
    updated_at,
    region,
    updated_type
)
VALUES %s
ON CONFLICT (branch_name, location_name) DO NOTHING;
"""

with conn.cursor() as cur:
    execute_values(cur, sql, records, page_size=1000)

conn.commit()

print(f"Inserted {len(records)} new branch/location combinations.")

Inserted 1 new branch/location combinations.


In [ ]:
df[df["ActivationKey"].notna()]

In [ ]:
# Keep only rows with ActivationKey
activation_df = df[df["ActivationKey"].notna()].copy()

# Validate ActivationKey format: XXXX-XXXX-XXXX-XXXX
pattern = re.compile(r'^[A-Za-z0-9]{4}(?:-[A-Za-z0-9]{4}){3}$')

activation_df = activation_df[
    activation_df["ActivationKey"].astype(str).str.match(pattern)
].copy()

# Activate machine
activation_df["IsActivated"] = 1

# Default License Type
activation_df["LicenseType"] = "Standard"

# Default Customer Plant ID
activation_df["CustPlantID"] = "PLANT-01"

# Convert DispatchDate to datetime
activation_df["DispatchDate"] = pd.to_datetime(
    activation_df["DispatchDate"], errors="coerce"
)

# Fill YearOfManufacturing from DispatchDate if missing or 0
mask = (
    activation_df["YearOfManufacturing"].isna()
    | (activation_df["YearOfManufacturing"] == 0)
)

activation_df.loc[mask, "YearOfManufacturing"] = (
    activation_df.loc[mask, "DispatchDate"].dt.year.astype("Int64")
)

# Keep only required columns
activation_df = activation_df[
    [
        "CNCID",
        "ActivationKey",
        "IsActivated",
        "LicenseType",
        "SupportNumber",
        "YearOfManufacturing",
        "WarrantyExpireDate",
        "CustAssetNumber",
        "CustPlantID",
        "ActivationAck",
        "AwarenessTrainingGivenOrNot",
        "MachineActivationDate"
    ]
]

activation_df

In [ ]:
# Fetch CNCID, Dispatch ID and Customer Name from DB
query = """
SELECT
    fmdl.cnc_id,
    fmdl.dispatch_id,
    dc.customer_name
FROM sap.fact_machine_dispatch_logs fmdl
INNER JOIN sap.dim_customer dc
    ON fmdl.customer_id = dc.customer_id
WHERE fmdl.job_id in(40);
"""

with conn.cursor() as cur:
    cur.execute(query)
    rows = cur.fetchall()
    columns = [desc[0] for desc in cur.description]

customer_df = pd.DataFrame(rows, columns=columns)

# Rename for merge
customer_df.rename(columns={"cnc_id": "CNCID"}, inplace=True)

# Remove duplicates
customer_df = customer_df.drop_duplicates(subset=["CNCID"])

# Remove existing columns if present
activation_df = activation_df.drop(
    columns=["customer_name", "dispatch_id"],
    errors="ignore"
)

# Merge
activation_df = activation_df.merge(
    customer_df,
    on="CNCID",
    how="left"
)

activation_df["ActivatedByEmailID"] = "supportteam@gmail.com"

activation_df

In [ ]:
activation_df.to_csv('28072026_cleanups/12092026_cleanups_v1/06_activation_records_25082026.csv',index=False,quoting=csv.QUOTE_ALL)

In [ ]:
results = []

sql = """
SELECT *
FROM miot_support.activate_cnc_id(
    %s,  -- p_cnc_id
    %s,  -- p_activation_key
    %s,  -- p_is_activated
    %s,  -- p_license_type
    %s,  -- p_support_number
    %s,  -- p_year_of_manufacturing
    %s,  -- p_warranty_date
    %s,  -- p_customer_assest_number
    %s,  -- p_customer_plant_id
    %s,  -- p_activated_by_email_id
    %s   -- p_customer_name
);
"""

total_rows = len(activation_df)

with conn.cursor() as cur:

    for idx, (_, row) in enumerate(activation_df.iterrows(), start=1):

        cur.execute(
            sql,
            (
                str(row["CNCID"]),
                str(row["ActivationKey"]),
                bool(row["IsActivated"]),
                str(row["LicenseType"]),
                None if pd.isna(row["SupportNumber"]) else str(row["SupportNumber"]),
                None if pd.isna(row["YearOfManufacturing"]) else int(row["YearOfManufacturing"]),
                None if pd.isna(row["WarrantyExpireDate"]) else pd.to_datetime(row["WarrantyExpireDate"]).date(),
                None if pd.isna(row["CustAssetNumber"]) else str(row["CustAssetNumber"]),
                str(row["CustPlantID"]),
                str(row["ActivatedByEmailID"]),
                None if pd.isna(row["customer_name"]) else str(row["customer_name"]),
            ),
        )

        v_msg, v_is_error = cur.fetchone()

        results.append(
            {
                "dispatch_id": row["dispatch_id"],
                "v_msg": v_msg,
                "v_is_error": v_is_error,
            }
        )

        # Print progress
        print(f"Processed {idx}/{total_rows}", flush=True)

conn.commit()

# Create result DataFrame
result_df = pd.DataFrame(results)

print("\nCompleted processing all rows.")
result_df

In [ ]:
result_df[result_df['v_is_error'] == False]['v_msg'][0]

In [ ]:
dispatch_update_df = activation_df[
    ["dispatch_id", "MachineActivationDate", "YearOfManufacturing"]
].copy()

dispatch_update_df["YearOfManufacturing"] = (
    dispatch_update_df["YearOfManufacturing"]
    .astype("Int64")   # Nullable integer dtype
)

dispatch_update_df.columns = [
    "dispatch_id",
    "machine_activation_date",
    "year_of_manufacturing",
]

# Create TSV in memory
buffer = io.StringIO()
dispatch_update_df.to_csv(buffer, sep="\t", index=False, header=False, na_rep="\\N")
buffer.seek(0)

with conn.cursor() as cur:
    cur.execute("""
        CREATE TEMP TABLE temp_dispatch_update (
            dispatch_id INTEGER,
            machine_activation_date TIMESTAMP,
            year_of_manufacturing INTEGER
        ) ON COMMIT DROP;
    """)

    cur.copy_from(
        buffer,
        "temp_dispatch_update",
        columns=(
            "dispatch_id",
            "machine_activation_date",
            "year_of_manufacturing",
        ),
    )

    cur.execute("""
        UPDATE sap.fact_machine_dispatch_logs f
        SET
            machine_activation_date = t.machine_activation_date,
            year_of_manufacturing = t.year_of_manufacturing,
            updated_at = NOW()
        FROM temp_dispatch_update t
        WHERE f.dispatch_id = t.dispatch_id;
    """)

conn.commit()

print(f"Updated {len(dispatch_update_df)} rows successfully.")

In [ ]:
# out_of = pd.read_csv(
#     '05_dq_out_of_kmat_cnc.csv',
#     delimiter=",",
#     encoding='utf-8'
# )

# supplier_slno_count = (
#     out_of.groupby("SupplierProduct")["CustomerProductSlno"]
#       .nunique()
#       .reset_index(name="DistinctCustomerProductSlno")
#       .sort_values("DistinctCustomerProductSlno", ascending=False)
# )

# supplier_slno_count